In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import Dict, List
import json

from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator, ModelError, Plotter
from presidio_evaluator.experiment_tracking import get_experiment_tracker

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2


dataset_name = "data_person_1000_zh_fixed_id.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd(), "data", dataset_name), token_model_version="zh_core_web_sm")
print(len(dataset))

def get_entity_counts(dataset: List[InputSample]) -> Dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter


entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print("\nMin and max number of tokens in dataset: "\
f"Min: {min([len(sample.tokens) for sample in dataset])}, "\
f"Max: {max([len(sample.tokens) for sample in dataset])}")

print(f"Min and max sentence length in dataset: " \
f"Min: {min([len(sample.full_text) for sample in dataset])}, "\
f"Max: {max([len(sample.full_text) for sample in dataset])}")

/home/capcom/anaconda3/envs/presidio-research/lib/python3.10/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/capcom/anaconda3/envs/presidio-research/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


stanza and spacy_stanza are not installed
Flair is not installed by default


tokenizing input:   0%|          | 0/975 [00:00<?, ?it/s]

loading model zh_core_web_sm


tokenizing input: 100%|██████████| 975/975 [00:11<00:00, 82.75it/s]


975
Count per entity:
[('O', 79696), ('STREET_ADDRESS', 5497), ('PERSON', 3394),
 ('EMAIL_ADDRESS', 3385), ('AGE', 1925), ('PHONE_NUMBER', 974),
 ('CREDIT_CARD', 53)]

Min and max number of tokens in dataset: Min: 77, Max: 126
Min and max sentence length in dataset: Min: 180, Max: 259


In [2]:
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider
from hanlp_engine import HanLPNlpEngine, HanLPRecognizer
import hanlp

tok = hanlp.load(hanlp.pretrained.tok.COARSE_ELECTRA_SMALL_ZH)
ner = hanlp.load(hanlp.pretrained.ner.MSRA_NER_ELECTRA_SMALL_ZH)
hanlp_model = hanlp.pipeline().append(tok, output_key="tok").append(
    ner, input_key="tok", output_key="ner"
)

nlp_engine = HanLPNlpEngine(hanlp_model)
recognizer = HanLPRecognizer(nlp_engine)

registry = RecognizerRegistry(supported_languages=["zh"])
registry.add_recognizer(recognizer)

from presidio_analyzer import PatternRecognizer, Pattern

email_pattern = Pattern(
    name="email",
    regex=r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    score=0.85,
)


email_recognizer = PatternRecognizer(
    supported_entity="EMAIL_ADDRESS",
    patterns=[email_pattern],
    name="EmailRecognizer",
    supported_language="zh",
)

registry.add_recognizer(email_recognizer)

phone_pattern = Pattern(
    name="cn_phone",
    regex=r"(?<!\d)(?:\+?86[- ]?)?1[3-9]\d{9}(?!\d)",
    score=0.9,
)

phone_recognizer = PatternRecognizer(
    supported_entity="PHONE_NUMBER",
    patterns=[phone_pattern],
    name="PhoneRecognizer",
    supported_language="zh",
)

registry.add_recognizer(phone_recognizer)

id_pattern = Pattern(
    name="cn_id",
    regex=r"(?<!\d)\d{17}[\dXx](?!\d)",
    score=0.95,
)

id_recognizer = PatternRecognizer(
    supported_entity="ID",
    patterns=[id_pattern],
    name="IdRecognizer",
    supported_language="zh",
)

registry.add_recognizer(id_recognizer)

analyzer_engine = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["zh"], registry=registry)

pprint(f"Supported entities for Chinese:")
pprint(analyzer_engine.get_supported_entities("zh"), compact=True)

print(f"\nLoaded recognizers for Chinese:")
pprint([rec.name for rec in analyzer_engine.registry.get_recognizers("zh", all_fields=True)], compact=True)

print(f"\nLoaded NER models:")
# pprint(analyzer_engine.nlp_engine.models)

'Supported entities for Chinese:'
['PERSON', 'EMAIL_ADDRESS', 'AGE', 'GENDER', 'PHONE_NUMBER', 'LOCATION', 'ID']

Loaded recognizers for Chinese:
['HanLPRecognizer', 'EmailRecognizer', 'PhoneRecognizer', 'IdRecognizer']

Loaded NER models:


In [3]:
from presidio_evaluator.models import PresidioAnalyzerWrapper

entities_mapping=PresidioAnalyzerWrapper.presidio_entities_map # default mapping
# Add titles and zip codes as we have recognizers for those
entities_mapping["GENDER"] = "GENDER"
# entities_mapping["STREET_ADDRESS"] = "LOCATION"
print("Entities mapping:")
pprint(entities_mapping)

dataset = SpanEvaluator.align_entity_types(
    dataset, 
    entities_mapping=entities_mapping, 
    allow_missing_mappings=True
)
new_entity_counts = get_entity_counts(dataset)
print("\nCount per entity after alignment:")
pprint(new_entity_counts.most_common(), compact=True)

dataset_entities = list(new_entity_counts.keys())

Entities mapping:
{'ADDRESS': 'LOCATION',
 'AGE': 'AGE',
 'BIRTHDAY': 'DATE_TIME',
 'CITY': 'LOCATION',
 'CREDIT_CARD': 'CREDIT_CARD',
 'CREDIT_CARD_NUMBER': 'CREDIT_CARD',
 'DATE': 'DATE_TIME',
 'DATE_OF_BIRTH': 'DATE_TIME',
 'DATE_TIME': 'DATE_TIME',
 'DOB': 'DATE_TIME',
 'DOMAIN': 'URL',
 'DOMAIN_NAME': 'URL',
 'EMAIL': 'EMAIL_ADDRESS',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'FACILITY': 'LOCATION',
 'FIRST_NAME': 'PERSON',
 'GENDER': 'GENDER',
 'GPE': 'LOCATION',
 'HCW': 'PERSON',
 'HOSP': 'ORGANIZATION',
 'HOSPITAL': 'ORGANIZATION',
 'IBAN': 'IBAN_CODE',
 'IBAN_CODE': 'IBAN_CODE',
 'ID': 'ID',
 'IP_ADDRESS': 'IP_ADDRESS',
 'LAST_NAME': 'PERSON',
 'LOC': 'LOCATION',
 'LOCATION': 'LOCATION',
 'NAME': 'PERSON',
 'NATIONALITY': 'NRP',
 'NORP': 'NRP',
 'NRP': 'NRP',
 'O': 'O',
 'ORG': 'ORGANIZATION',
 'ORGANIZATION': 'ORGANIZATION',
 'PATIENT': 'PERSON',
 'PATORG': 'ORGANIZATION',
 'PER': 'PERSON',
 'PERSON': 'PERSON',
 'PHONE': 'PHONE_NUMBER',
 'PHONE_NUMBER': 'PHONE_NUMBER',
 'PREFIX': '

In [4]:
text = "贾明川，38岁男性建筑设计师，现居住于黑龙江省哈尔滨市道里区中央大街123号。其身份证号码为230102198511153216，可通过邮箱jiamingchuan@163.com或手机13945678231联系。贾先生近期出现体重超标、呼吸短促和关节不适等症状，经哈尔滨医科大学附属第一医院诊断确诊为肥胖症。主治医师金志明为其开具了扑热息痛作为治疗方案。贾先生目前信用评分为549分，年收入100245.67元人民币，主要使用中国工商银行账户（卡号：6222021005678912345）进行资金往来。"
results = analyzer_engine.analyze(text=text, language="zh", return_decision_process=True)
for result in results:
    print(f"\nEntity: {result.entity_type}, Text: {text[result.start:result.end]}\n\nAnalysis explanation:")
    pprint(result.analysis_explanation)

print(dataset[0].full_text, results)


Entity: ID, Text: 230102198511153216

Analysis explanation:
{'recognizer': 'IdRecognizer', 'pattern_name': 'cn_id', 'pattern': '(?<!\\d)\\d{17}[\\dXx](?!\\d)', 'original_score': 0.95, 'score': 0.95, 'textual_explanation': 'Detected by `IdRecognizer` using pattern `cn_id`', 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': None, 'regex_flags': regex.I|regex.M|regex.S}

Entity: PHONE_NUMBER, Text: 13945678231

Analysis explanation:
{'recognizer': 'PhoneRecognizer', 'pattern_name': 'cn_phone', 'pattern': '(?<!\\d)(?:\\+?86[- ]?)?1[3-9]\\d{9}(?!\\d)', 'original_score': 0.9, 'score': 0.9, 'textual_explanation': 'Detected by `PhoneRecognizer` using pattern `cn_phone`', 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': None, 'regex_flags': regex.I|regex.M|regex.S}

Entity: PERSON, Text: 贾明川

Analysis explanation:
None

Entity: AGE, Text: 38岁

Analysis explanation:
None

Entity: LOCATION, Text: 黑龙江省

Analysis explanation:
Non

In [5]:
# Set up the experiment tracker to log the experiment for reproducibility
experiment = get_experiment_tracker()


from hanlp_model_wrapper import HanLPModelWrapper

model = HanLPModelWrapper(
    analyzer_engine=analyzer_engine,
    labeling_scheme="IO",
    language="zh",
)

# Create the evaluator object
evaluator = SpanEvaluator(model=model, iou_threshold=0.7)


# Track model and dataset params
params = {"dataset_name": dataset_name, "model_name": evaluator.model.name}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)
experiment.log_parameter("entity_mappings", json.dumps(entities_mapping))

## Run experiment

evaluation_results = evaluator.evaluate_all(dataset, language="zh")
results = evaluator.calculate_score(evaluation_results)

# Track experiment results
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, 
                                labels=entities)

# end experiment
experiment.end()

# Note that the experiment params and metrics are saved locally

/home/capcom/presidio/presidio-research/presidio_evaluator/evaluation/base_evaluator.py:83: UserWarning: skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]
  warnings.warn("skip words not provided, using default skip words. "


Running model HanLPModelWrapper on dataset...
Finished running model on dataset
saving experiment data to /home/capcom/presidio/Qwen3Guard/experiment_20260319-211328.json


In [6]:
from collections import Counter

predicted_label_counts = Counter(
    tag
    for r in evaluation_results
    for tag in r.predicted_tags
    if tag != "O"
)

print(predicted_label_counts)


Counter({'LOCATION': 5713, 'EMAIL_ADDRESS': 3258, 'PERSON': 2501, 'AGE': 1925, 'PHONE_NUMBER': 1282, 'ID': 975})


In [7]:
# # Plot output
plotter = Plotter(results=results, 
                  model_name = evaluator.model.name, 
                  save_as="svg",
                  beta = 2) 

# Path of the directory to save the plots
output_folder = Path(Path.cwd(), "plotter_output")
plotter.plot_scores(output_folder=output_folder)

In [8]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('邮箱', 59),
 ('中国 工商 银行', 54),
 ('6222021234567890123', 10),
 ('邮件', 10),
 ('中国 工商 银行 杭州 分行', 7),
 ('6222021208001234567', 6),
 ('6222020302012345678', 6),
 ('阿司匹林', 6),
 ('6222020302056789123', 6),
 ('230102195001011234', 5)]
---------------
Example sentence with each FP token:
	- 邮箱 (`邮箱` pred as PHONE_NUMBER)
	- 中国 工商 银行 (`中国 工商 银行` pred as LOCATION)
	- 6222021234567890123 (`6222021234567890123` pred as PHONE_NUMBER)
	- 邮件 (`邮件` pred as EMAIL_ADDRESS)
	- 中国 工商 银行 杭州 分行 (`中国 工商 银行 杭州 分行` pred as LOCATION)
	- 6222021208001234567 (`6222021208001234567` pred as PHONE_NUMBER)
	- 6222020302012345678 (`6222020302012345678` pred as PHONE_NUMBER)
	- 阿司匹林 (`阿司匹林` pred as LOCATION)
	- 6222020302056789123 (`6222020302056789123` pred as PHONE_NUMBER)
	- 230102195001011234 (`230102195001011234` pred as ID)


[('邮箱', 59),
 ('中国 工商 银行', 54),
 ('6222021234567890123', 10),
 ('邮件', 10),
 ('中国 工商 银行 杭州 分行', 7),
 ('6222021208001234567', 6),
 ('6222020302012345678', 6),
 ('阿司匹林', 6),
 ('6222020302056789123', 6),
 ('230102195001011234', 5)]

In [13]:
fps_df = ModelError.get_fps_dataframe(results.model_errors, entity=["PHONE_NUMBER"])
fps_df[["full_text", "token", "annotation", "prediction", "sample_id"]].head(20)

,full_text,token,annotation,prediction,sample_id
0,邮箱,邮箱,O,PHONE_NUMBER,None
1,6222020200001234567,6222020200001234567,O,PHONE_NUMBER,None
2,keyaq in @ 163.com 13945671234,keyaq 163.com 13945671234,O,PHONE_NUMBER,None
3,keyaq in @ 163.com 13945671234,keyaq 163.com 13945671234,O,PHONE_NUMBER,None
4,6222021234567890123,6222021234567890123,O,PHONE_NUMBER,None
5,6222021208001234567,6222021208001234567,O,PHONE_NUMBER,None
6,6222020302012345678,6222020302012345678,O,PHONE_NUMBER,None
7,6222021005678912345,6222021005678912345,O,PHONE_NUMBER,None
8,STL31071,stl31071,O,PHONE_NUMBER,None
9,6222020308123456789,6222020308123456789,O,PHONE_NUMBER,None


In [10]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)

Most common false negative tokens:
[('沈雨桐', 40),
 ('崔明远', 40),
 ('阮雅琴', 33),
 ('柴明 远', 24),
 ('6222021234567890123', 12),
 ('韩雅琴', 10),
 ('13812345678', 10),
 ('韩明 远', 9),
 ('沈雅琴', 9),
 ('张建国', 9),
 ('6222020302012345678', 8),
 ('张雪梅', 7),
 ('6222021208001234567', 7),
 ('阮志明', 7),
 ('6222020200001234567', 6)]
---------------
Example sentence with each FN token:
	- 沈雨桐 (`沈雨桐` annotated as PERSON)
	- 崔明远 (`崔明远` annotated as PERSON)
	- 阮雅琴 (`阮雅琴` annotated as PERSON)
	- 柴明 远 (`柴明 远` annotated as PERSON)
	- 6222021234567890123 (`6222021234567890123` annotated as CREDIT_CARD)
	- 韩雅琴 (`韩雅琴` annotated as PERSON)
	- 13812345678 (`13812345678` annotated as PHONE_NUMBER)
	- 韩明 远 (`韩明 远` annotated as PERSON)
	- 沈雅琴 (`沈雅琴` annotated as PERSON)
	- 张建国 (`张建国` annotated as PERSON)
	- 6222020302012345678 (`6222020302012345678` annotated as CREDIT_CARD)
	- 张雪梅 (`张雪梅` annotated as PERSON)
	- 6222021208001234567 (`6222021208001234567` annotated as CREDIT_CARD)
	- 阮志明 (`阮志明` annotated as PERSON)
	- 622202

[('沈雨桐', 40),
 ('崔明远', 40),
 ('阮雅琴', 33),
 ('柴明 远', 24),
 ('6222021234567890123', 12),
 ('韩雅琴', 10),
 ('13812345678', 10),
 ('韩明 远', 9),
 ('沈雅琴', 9),
 ('张建国', 9),
 ('6222020302012345678', 8),
 ('张雪梅', 7),
 ('6222021208001234567', 7),
 ('阮志明', 7),
 ('6222020200001234567', 6)]

In [11]:
fns_df = ModelError.get_fns_dataframe(results.model_errors, entity=["PERSON"])
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,白雅宁,白雅宁,PERSON,O
1,陶立 轩,陶立 轩,PERSON,O
2,林雨 晴,林雨 晴,PERSON,O
3,林雨 晴 张雪梅 林雨 晴年,林雨 晴 张雪梅 林雨 晴年,PERSON,O
4,柏晓岚,柏晓岚,PERSON,O
5,贾思明,贾思明,PERSON,O
6,阮雅雯,阮雅雯,PERSON,O
7,安茹薇,安茹薇,PERSON,O
8,钟立诚,钟立诚,PERSON,O
9,钟立诚 张伟明 钟立 诚年,钟立诚 张伟明 钟立 诚年,PERSON,O
